# 05 · Grad-CAM, multi-layer Grad-CAM, error analysis, final report
> **Research prototype.** Predicts *dataset-defined* dementia categories from preprocessed JPG brain MRI slices. Not a clinically validated diagnostic system.

Steps 26–29 and 33–35. Standard Grad-CAM is verified first (target layer found from measured feature-map shapes;
activation / gradient present and non-empty, no NaN, correct size, correct target, consistent model output) for the
predicted class and for every manually selected class. Then multi-layer Grad-CAM (early / middle / late / final +
fused), sanity checks (parameter randomisation, deletion test), error analysis with Grad-CAM, the final report, the
model package and a `predict.py` demonstration.

**Grad-CAM is an attribution technique — not a lesion segmentation, a biomarker, proof of pathology or a diagnosis.**

In [ ]:
# Bootstrap: locate the project code (local checkout, /kaggle/working, or a Kaggle dataset upload)
import os, sys, glob, shutil, json
from pathlib import Path

CANDIDATES = [Path.cwd().parent, Path.cwd(), Path("/kaggle/working/dementia_mri_project")]
CANDIDATES += [Path(p).parent for p in glob.glob("/kaggle/input/*/dementia_mri_project/src")
               + glob.glob("/kaggle/input/*/*/dementia_mri_project/src") + glob.glob("/kaggle/input/*/src")]
CODE = next((c for c in CANDIDATES if (c / "src" / "run_pipeline.py").exists()), None)
if CODE is None:
    raise FileNotFoundError("Project code not found. On Kaggle: upload dementia_mri_project.zip as a Dataset and add it "
                            "as an Input (Kaggle extracts it), or put the project folder next to this notebook.")
if str(CODE).startswith("/kaggle/input"):                       # /kaggle/input is read-only -> work in /kaggle/working
    dst = Path("/kaggle/working/dementia_mri_project")
    shutil.copytree(CODE / "src", dst / "src", dirs_exist_ok=True)
    shutil.copy(CODE / "predict.py", dst / "predict.py")
    if not (dst / "configs" / "config.json").exists():
        shutil.copytree(CODE / "configs", dst / "configs", dirs_exist_ok=True)
    for sub in ("data", "checkpoints", "results", "gradcam", "reports"):   # outputs of earlier notebooks, if attached
        if (CODE / sub).exists() and not (dst / sub).exists():
            shutil.copytree(CODE / sub, dst / sub)
    CODE = dst
sys.path.insert(0, str(CODE)); os.chdir(CODE)
print("Project code:", CODE)

from src.utils import load_config, get_paths, print_env, load_json, save_json
from src.nb_helpers import show, show_md, show_table, sample_grid

CONFIG_OVERRIDES = {}      # e.g. {"batch_size": 16, "num_workers": 4, "epochs_stage2": 30}
cfg = load_config(overrides=CONFIG_OVERRIDES)
paths = get_paths(cfg)
ENV = print_env()
print("Outputs ->", paths["root"])

In [ ]:
from src.run_pipeline import stage_gradcam
CAM = stage_gradcam(cfg, paths)
print(json.dumps({k: CAM[k] for k in ("layers", "fusion_method", "validity", "sanity")}, indent=1, default=str))

## Standard Grad-CAM: predicted class and manually selected target classes

In [ ]:
for p in sorted((paths["gradcam"] / "standard").glob("*_panel.png")):
    print(p.name); show(p)

## Multi-layer Grad-CAM examples per class

In [ ]:
for cdir in sorted(d for d in paths["gradcam"].iterdir() if d.is_dir() and d.name not in ("standard", "errors")):
    for p in sorted(cdir.glob("*_panel.png"))[:2]:
        print(cdir.name, "|", p.name); show(p)

## Error analysis

In [ ]:
import pandas as pd
print(json.dumps(load_json(paths["results"] / "error_analysis_summary.json"), indent=1, default=str))
show_table(paths["results"] / "error_analysis.csv", 15)
for g in ("confident_incorrect", "low_confidence"):
    for p in sorted((paths["gradcam"] / "errors" / g).glob("*_panel.png"))[:3]:
        print(g, "|", p.name); show(p)

## Final report and model package

In [ ]:
from src.run_pipeline import stage_report, stage_package
stage_report(cfg, paths); stage_package(cfg, paths)
show_md(paths["root"] / "FINAL_REPORT.md")

## Single-image inference with `predict.py`

In [ ]:
import subprocess
test = pd.read_csv(paths["data"] / "test.csv")
img = test.sample(1, random_state=0).filepath.iloc[0]
r = subprocess.run([sys.executable, "predict.py", "--image", img, "--checkpoint", str(paths["checkpoints"] / "best_model.pth"),
                    "--output-dir", str(paths["root"] / "predictions"), "--mc-dropout", "20"], capture_output=True, text=True)
print(r.stdout[-4000:], r.stderr[-2000:])